# Análise de dosagem de insumos por batelada

Notebook de análise estatística da dosagem automática de insumos (farinha, gordura, água) e temperatura de uma massa, com base nas bateladas registradas pelo sistema.

Inclui: distribuição e outliers, série temporal, carta de controle, análise de capacidade (Cp/Cpk) e ramp-up das primeiras bateladas.

> **Os dados em `data/bateladas_exemplo.xlsx` são sintéticos**, gerados só para demonstração. Nenhum valor representa receita, equipamento ou resultado real de empresa.

Colunas: `_s` = setpoint (valor programado) e `_r` = valor realizado.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

## 1. Carga e preparação dos dados

In [ ]:
CAMINHO = "data/bateladas_exemplo.xlsx"

df = pd.read_excel(CAMINHO, sheet_name="BD")

# Renomear colunas (_s = setpoint, _r = realizado)
df.columns = [
    "Batelada", "Farinha_s", "Farinha_r",
    "Gordura_s", "Gordura_r",
    "Agua_s", "Agua_r", "Temp_s", "Temp_r",
]

df = df.dropna(subset=["Batelada"])  # remove linhas totalmente vazias no fim

# Colunas numéricas de interesse para análise
colunas_numericas = [
    "Farinha_r",
    "Gordura_r",
    "Agua_r", "Temp_r",
]

print("Shape:", df.shape)
print("\nValores nulos por coluna:\n", df[colunas_numericas].isna().sum())
print("\nEstatísticas descritivas:\n", df[colunas_numericas].describe())

## 2. Distribuição, outliers e série por batelada

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 10))
axes = axes.flatten()
 
for i, col in enumerate(colunas_numericas):
    axes[i].hist(df[col].dropna(), bins=8, edgecolor="black", alpha=0.6, density=True)
    sns.kdeplot(df[col].dropna(), ax=axes[i], color="red", linewidth=2)
    axes[i].set_title(col)
 
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 10))
axes = axes.flatten()
 
for i, col in enumerate(colunas_numericas):
    axes[i].boxplot(df[col].dropna(), vert=True)
    axes[i].set_title(col)
 
plt.tight_layout()
plt.savefig("boxplot_insumos.png", dpi=120)
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 10))
axes = axes.flatten()
 
variaveis_serie = [
    "Farinha_r",
    "Gordura_r",
    "Agua_r", "Temp_r",
]

for i, col in enumerate(variaveis_serie):
    axes[i].plot(df["Batelada"], df[col], marker="o", markersize=2, linewidth=0.8)
    axes[i].set_title(f"{col} por batelada")
    axes[i].set_xlabel("Batelada")
 
plt.tight_layout()
plt.show()

## 3. Carta de controle (Farinha)

In [ ]:
col_controle = "Farinha_r" # variável de interesse
media = df["Farinha_r"].mean()
desvio = df[col_controle].std()

print(media) 
print(f'{(media + (media * 0.01))}')
print(f'{(media - (media * 0.01))}')
plt.figure(figsize=(12, 5))
plt.plot(df["Batelada"], df[col_controle], marker="o", markersize=3, linewidth=0.8)
plt.axhline(media, color="green", label="Média")
plt.axhline((media + (media * 0.01)), color="red", linestyle="--", label="LSC (+1%)")
plt.axhline((media - (media * 0.01)), color="red", linestyle="--", label="LIC (-1%)")
plt.title(f"Carta de controle - {col_controle}")
plt.xlabel("Batelada")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Análise de capacidade (Farinha)

In [ ]:
from scipy.stats import norm

# Alvo e limites de especificação (alvo ± 1%)
d2 = df['Farinha_r'].mean()
LIE = d2 - (d2 * 0.01)
LSE = d2 + (d2 * 0.01)


media_geral = df['Farinha_r'].mean()
r_barra = df['Farinha_s'].mean()
sigma_dentro = r_barra / d2

dados = df[['Farinha_r']].values.flatten()

fig, ax = plt.subplots(figsize=(10, 8))

n_bins = 10
ax.hist(dados, bins=n_bins, alpha=0.6, color='steelblue', edgecolor='black')

x = np.linspace(LIE - 5, LSE + 5, 50)
largura_bin = (dados.max() - dados.min()) / n_bins
curva = norm.pdf(x, media_geral, sigma_dentro) * len(dados) * largura_bin
ax.plot(x, curva, 'k--', linewidth=1.5, label='Curva Normal (Dentro)')

ax.axvline(LIE, color='red', linestyle='--', label='LIE')
ax.axvline(LSE, color='red', linestyle='--', label='LSE')
ax.axvline(media_geral, color='green', linestyle='-', label='Média')

ax.set_title('Análise de Capacidade - Farinha')
ax.set_xlabel('')
ax.set_ylabel('')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Cp e Cpk por insumo

In [ ]:
# Tolerância (%) em torno do alvo de cada insumo
tolerancias = {
    "Farinha_r": 0.01,
    "Gordura_r": 0.005,
    "Agua_r":    0.005,
}

# Limites de especificação: (LSL, USL) = alvo -/+ tolerância
especificacoes = {
    col: (df[col].mean() * (1 - tol), df[col].mean() * (1 + tol))
    for col, tol in tolerancias.items()
}

def calcular_cp_cpk(coluna, LSL, USL):
    media = coluna.mean()
    desvio = coluna.std(ddof=1)
    cp = (USL - LSL) / (6 * desvio)
    cpk = min((USL - media) / (3 * desvio), (media - LSL) / (3 * desvio))
    return cp, cpk, media, desvio

resultados = []
for col, (lsl, usl) in especificacoes.items():
    if lsl is None or usl is None:
        continue
    cp, cpk, media, desvio = calcular_cp_cpk(df[col].dropna(), lsl, usl)
    resultados.append({
        "Insumo": col, "Média": round(media, 3), "Desvio": round(desvio, 3),
        "LSL": lsl, "USL": usl, "Cp": round(cp, 3), "Cpk": round(cpk, 3)
    })

tabela = pd.DataFrame(resultados)
print(tabela)

## 6. Desvio em relação ao setpoint e ramp-up

In [ ]:
df['Agua_dv%'] = (df['Agua_r'] / df['Agua_s'] - 1) * 100
df['Farinha_dv%'] = (df['Farinha_r'] / df['Farinha_s'] - 1) * 100
df['Gordura_dv%'] = (df['Gordura_r'] / df['Gordura_s'] - 1 )* 100

df.head()

In [ ]:

df30 = df.iloc[:30]

plt.figure(figsize=(12, 5))
plt.plot(df30['Batelada'], df30['Agua_dv%'], color='orange', marker="o", markersize=3, linewidth=1.5, label="Água")
plt.plot(df30['Batelada'], df30['Farinha_dv%'], color='blue', marker= 6, markersize=3, linewidth=1.5, label="Farinha")
plt.plot(df30['Batelada'], df30['Gordura_dv%'], color='brown', marker="s", markersize=3, linewidth=1.5, label="Gordura")
plt.title("Ramp-up sistemas de dosagens 30 primeiras bateladas (%)")
plt.xlabel("Batelada")
plt.grid(linestyle=':')
plt.legend()
plt.tight_layout()
plt.show()